# StreamFM Speech Enhancement Baseline

This notebook establishes a small reproducible baseline for `streamfm_se_predgen`: offline inference on a fixed clip, then frame-step timing for the DNN streaming core.

## 1. Setup
The helper functions live in `experiments/baseline/streamfm_se_baseline.py` so this notebook stays readable.

In [9]:
from pathlib import Path
import os
import sys
import pandas as pd
import torch
import torchaudio

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'config').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from experiments.baseline.streamfm_se_baseline import (
    benchmark_frame_steps,
    load_mono_audio,
    load_streamfm_se_model,
    run_offline_inference,
    select_device,
    write_results,
)

print('repo:', REPO_ROOT)
print('torch:', torch.__version__)
print('mps available:', torch.backends.mps.is_available())

repo: /Users/damien/Documents/HW/Research_project/streamfm
torch: 2.7.0
mps available: True


## 2. Parameters

In [10]:
DEVICE_NAME = 'mps'  # use 'cpu' if needed
SOLVER = 'euler'
STEPS = 1
FRAME_STEPS_TO_TEST = [1]  # later you can try [1, 2, 5]

AUDIO_PATH = REPO_ROOT / 'inputs/test_clips/audio_43m28_10s.wav'
OUT_DIR = REPO_ROOT / 'outputs/baseline_se'
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = select_device(DEVICE_NAME)
device

device(type='mps')

## 3. Load StreamFM SE

In [11]:
model, cfg = load_streamfm_se_model(device=device)
print(type(model).__name__)
print('sampling rate:', model.sampling_rate)
print('predictor:', type(model.initial_predictor).__name__)
print('flow backbone:', type(model.dnn).__name__)

CompressedAmplitudeComplexSTFT initialized with 32.00ms window, 16.00ms hop. alpha=0.5, beta=1.0, compression_is_learnable=False, normalized_stft=True, amplitude_only=False.
Initialized AudioDataModule with batch size 12
DistillMOS variant 7 weights loaded from: /Users/damien/Documents/HW/Research_project/streamfm/.venv/lib/python3.10/site-packages/distillmos/weights/distill_mos_v7.pt
CompressedAmplitudeComplexSTFT initialized with 32.00ms window, 16.00ms hop. alpha=0.5, beta=1.0, compression_is_learnable=False, normalized_stft=True, amplitude_only=False.
Initialized AudioDataModule with batch size 12
DistillMOS variant 7 weights loaded from: /Users/damien/Documents/HW/Research_project/streamfm/.venv/lib/python3.10/site-packages/distillmos/weights/distill_mos_v7.pt


/Users/damien/Documents/HW/Research_project/streamfm/.venv/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'losses' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['losses'])`.


StoRMOnTheFlyFlowModel
sampling rate: 16000
predictor: DiscriminativeModel
flow backbone: CausalNCSNpp


## 4. Load Test Clip

In [12]:
wav, sr = load_mono_audio(AUDIO_PATH, device=device, target_sr=model.sampling_rate)
duration_s = wav.shape[-1] / sr
print('path:', AUDIO_PATH)
print('shape:', tuple(wav.shape))
print('sample rate:', sr)
print('duration:', round(duration_s, 3), 's')

path: /Users/damien/Documents/HW/Research_project/streamfm/inputs/test_clips/audio_43m28_10s.wav
shape: (1, 160000)
sample rate: 16000
duration: 10.0 s


## 5. Offline Baseline
This is not true streaming. It measures `model.enhance(...)` on the whole clip, which is useful as a reproducible baseline and an optimistic throughput reference.

In [13]:
offline = run_offline_inference(
    model=model,
    wav=wav,
    sr=sr,
    device=device,
    solver=SOLVER,
    steps=STEPS,
)

out_wav = offline.pop('output')
out_path = OUT_DIR / f'offline_{device.type}_{SOLVER}_N{STEPS}.wav'
torchaudio.save(out_path, out_wav, sample_rate=sr)

offline['output_path'] = str(out_path)
offline

{'mode': 'offline_enhance',
 'device': 'mps',
 'solver': 'euler',
 'steps': 1,
 'audio_duration_s': 10.0,
 'elapsed_s': 3.1533304580007098,
 'rtf': 0.315333045800071,
 'output_path': '/Users/damien/Documents/HW/Research_project/streamfm/outputs/baseline_se/offline_mps_euler_N1.wav'}

## 6. Frame-Step Baseline
This measures the DNN streaming core: initial predictor plus flow `forward_step`. It bypasses `torch.compile` by default because it is unstable on this Mac/MPS setup.

In [14]:
frame_results = benchmark_frame_steps(
    model=model,
    device=device,
    steps_list=FRAME_STEPS_TO_TEST,
    iterations=100,
    warmup=10,
    use_compiled=False,
)
frame_results

[{'mean_ms': 58.458491239944124,
  'p50_ms': 55.32179099827772,
  'p90_ms': 60.195708996616304,
  'p99_ms': 129.13670799753163,
  'mode': 'frame_step_predictor_plus_flow',
  'device': 'mps',
  'steps': 1,
  'iterations': 100,
  'warmup': 10,
  'compiled': False,
  'frame_budget_ms': 16.0,
  'budget_ratio_mean': 3.6536557024965077}]

## 7. Summary Table

In [15]:
rows = [offline] + frame_results
df = pd.DataFrame(rows)
df

,mode,device,solver,steps,audio_duration_s,elapsed_s,rtf,output_path,mean_ms,p50_ms,p90_ms,p99_ms,iterations,warmup,compiled,frame_budget_ms,budget_ratio_mean
0,offline_enhance,mps,euler,1,10.0,3.15333,0.315333,/Users/damien/Documents/HW/Research_project/st...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,frame_step_predictor_plus_flow,mps,NaN,1,NaN,NaN,NaN,NaN,58.458491,55.321791,60.195709,129.136708,100.0,10.0,False,16.0,3.653656


In [16]:
json_path = OUT_DIR / 'streamfm_se_baseline_results.json'
csv_path = OUT_DIR / 'streamfm_se_baseline_results.csv'
write_results(json_path, rows)
df.to_csv(csv_path, index=False)
print('wrote:', json_path)
print('wrote:', csv_path)

wrote: /Users/damien/Documents/HW/Research_project/streamfm/outputs/baseline_se/streamfm_se_baseline_results.json
wrote: /Users/damien/Documents/HW/Research_project/streamfm/outputs/baseline_se/streamfm_se_baseline_results.csv


## 8. Interpretation Notes
- Offline RTF below 1 means the model can process a complete file faster than real time, but it does not prove live streaming.
- Frame-step latency should be compared to the STFT hop budget: `256 / 16000 = 16 ms`.
- If predictor + flow takes more than 16 ms before STFT/iSTFT and audio I/O, the current configuration is not real-time streaming on this device.